# Day 44 — Trees and Density

## Hierarchical clustering · DBSCAN

**Boston University · Data Science Program**

---

Yesterday ended on two hard limits. K-Means made you **know `k` in advance**, and it
could only ever draw **round blobs**. Today is one method for each.

| | |
|---|---|
| 1 | Merge the closest pair over and over, and keep the whole tree |
| 2 | Cut that tree anywhere to get any `k` — from a single fit |
| 3 | Choose a linkage, and watch it change the answer completely |
| 4 | Cluster by density instead: core, border and **noise** |
| 5 | Break DBSCAN with `eps`, and again with uneven density |
| 6 | Put all four methods on four shapes and see that none of them wins |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time, warnings
warnings.simplefilter("ignore")

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import make_moons, make_blobs, make_circles

TEAL, CORAL, VIOLET, BLUE, AMBER = "#1F9E86", "#E05C45", "#8268B5", "#3E7CB1", "#F0A530"
MUTED, INK = "#7B8AA1", "#131F33"
COLS = [TEAL, CORAL, VIOLET, BLUE, AMBER, "#C96FA0", "#5BA87F"]

def show(X, labels=None, title="", ax=None):
    ax = ax or plt.gca()
    if labels is None:
        ax.scatter(X[:, 0], X[:, 1], c="#9AA8BE", s=18, edgecolors="white", linewidths=.4)
    else:
        for u in sorted(set(labels)):
            m = labels == u
            if u == -1:
                ax.scatter(X[m, 0], X[m, 1], c=MUTED, marker="x", s=22, linewidths=1.2)
            else:
                ax.scatter(X[m, 0], X[m, 1], c=COLS[u % 7], s=18,
                           edgecolors="white", linewidths=.4)
    ax.set_title(title, fontsize=11); ax.set_xticks([]); ax.set_yticks([])

# the same 240 customers as Day 43, so you can compare directly
def load_customers(seed=43):
    r = np.random.default_rng(seed)
    a = np.c_[r.normal(18.0, 3.1, 90), r.normal(240, 45, 90)]
    b = np.c_[r.normal(3.6, 1.2, 70), r.normal(255, 48, 70)]
    c = np.c_[r.normal(5.2, 1.7, 80), r.normal(790, 100, 80)]
    X = np.vstack([a, b, c])
    truth = np.r_[np.zeros(90), np.ones(70), np.full(80, 2)].astype(int)
    X[:, 0] = X[:, 0].clip(1, None); X[:, 1] = X[:, 1].clip(80, None)
    i = r.permutation(len(X))
    return X[i], truth[i]

X, TRUTH = load_customers()
Z = StandardScaler().fit_transform(X)
print("customers:", len(X))

---
# 1 · Merge the closest pair, over and over

**Agglomerative clustering**, in three lines:

1. Every point starts as its own cluster
2. Find the two closest clusters and merge them
3. Repeat until one cluster is left

With `n` points that is exactly `n − 1` merges. Nothing is ever told how many clusters
to look for.

In [ ]:
L = linkage(Z, method="ward")      # the whole tree, computed once

print("rows                ", len(Z))
print("merges recorded     ", L.shape[0])
print("n - 1               ", len(Z) - 1)
print()
print("each row of L is:  [cluster A, cluster B, distance merged at, size after]")
print()
print("the first three merges (the closest pairs in the whole dataset):")
print(np.round(L[:3], 4))
print()
print("the last three merges (joining whole groups together):")
print(np.round(L[-3:], 4))

Look at the `distance merged at` column. The first merges happen at distances of about
**0.006** — two customers who are nearly identical. The final merge happens at
**23.99**, joining two large groups that are nothing like each other.

That column is the whole story, and the dendrogram is just a picture of it.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
dendrogram(L, ax=ax, no_labels=True, color_threshold=11)
ax.axhline(11, color=CORAL, ls="--", lw=1.8)
ax.text(ax.get_xlim()[1] * .99, 11.6, "cut here", ha="right", fontsize=10,
        color=CORAL, fontweight="bold")
ax.set_ylabel("distance merged at"); ax.set_title("240 customers, one tree")
plt.tight_layout(); plt.show()

## 1.1 · One fit, every k

This is the selling point. The tree was built **once**. Now get any `k` you like out of
it, for free, by cutting at a different height.

In [ ]:
print(f"{'k':>3}{'agreement with the truth':>26}{'silhouette':>13}")
for k in range(2, 7):
    lab = fcluster(L, k, criterion="maxclust")
    print(f"{k:>3}{adjusted_rand_score(TRUTH, lab):>26.4f}"
          f"{silhouette_score(Z, lab):>13.4f}")

print()
km = KMeans(3, n_init=10, random_state=0).fit(Z)
print("for comparison, K-Means at k=3:", round(adjusted_rand_score(TRUTH, km.labels_), 4))

**Ward at k = 3 scores 1.000** — a perfect recovery of the three customer groups. K-Means
managed 0.975 on the same data.

## 1.2 · Where should you cut?

Look for a **big vertical gap** in the tree. A merge that happens at a much greater
height than the one before it is joining two things that do not belong together — so you
keep the clusters that existed just below it.

In [ ]:
heights = L[:, 2]
print(f"{'clusters left':>14}{'merge height':>15}{'jump from the one below':>26}")
for i in range(1, 7):
    h = heights[-i]
    jump = h - heights[-i-1]
    print(f"{i:>14}{h:>15.2f}{jump:>26.2f}")

Going from 3 clusters to 2 costs **12.54** — by far the biggest jump on the list. That is
the tree saying *do not make that merge*, which leaves you with three.

> Notice this needed no labels, no silhouette and no second fit. The shape of the tree
> alone suggested `k = 3`.

---
# 2 · Linkage — what does "closest" mean?

Two single points have an obvious distance. Two *groups* of points do not. **Linkage** is
how you define it, and it is the one real choice you make.

| linkage | distance between two groups is… |
|---|---|
| `ward` | whichever merge raises total spread the least |
| `complete` | the distance between their two **furthest** points |
| `average` | the average over every pair |
| `single` | the distance between their two **closest** points |

In [ ]:
for lk in ["ward", "complete", "average", "single"]:
    lab = AgglomerativeClustering(3, linkage=lk).fit(Z).labels_
    print(f"  {lk:9} agreement {adjusted_rand_score(TRUTH, lab):.4f}"
          f"   sizes {sorted(np.bincount(lab).tolist())}")

**All four give exactly the same answer.** When the groups really are separate blobs, the
choice of linkage does not matter at all.

Now try a shape where it matters enormously.

In [ ]:
XM, YM = make_moons(300, noise=0.06, random_state=0)

fig, axes = plt.subplots(1, 4, figsize=(14, 3.3))
for ax, lk in zip(axes, ["ward", "complete", "average", "single"]):
    lab = AgglomerativeClustering(2, linkage=lk).fit(XM).labels_
    a = adjusted_rand_score(YM, lab)
    show(XM, lab, f"{lk}   {a:.3f}", ax=ax)
plt.tight_layout(); plt.show()

From **0.197 to 1.000** on identical data — the single biggest swing you will see from one
keyword argument all course.

**Why single linkage wins here.** It only ever asks *"are these two groups touching
anywhere?"*, so a cluster can snake along a curve one point at a time.

**And why that same greediness is dangerous.** One stray point sitting between two real
groups is enough to chain them into one. Single linkage is spectacular on connected
shapes and catastrophic when the gaps between groups are not clean. **Ward is the
sensible default.**

## 2.1 · What it costs

To merge the closest pair you must know every pairwise distance — and there are
`n × (n − 1) / 2` of them.

In [ ]:
print(f"{'rows':>7}{'pairs to hold':>18}{'hierarchical':>15}{'k-means':>11}")
for n in [500, 1000, 2000, 4000]:
    Xb, _ = make_blobs(n_samples=n, centers=4, random_state=0)
    Zb = StandardScaler().fit_transform(Xb)
    t = time.perf_counter(); AgglomerativeClustering(4).fit(Zb)
    th = time.perf_counter() - t
    t = time.perf_counter(); KMeans(4, n_init=10, random_state=0).fit(Zb)
    tk = time.perf_counter() - t
    print(f"{n:>7}{n*(n-1)//2:>18,}{th:>14.3f}s{tk:>10.3f}s")

print()
print(f"at 40,000 rows that would be {40000*39999//2:,} pairs")

Eight times the rows makes hierarchical clustering roughly **70× slower**, while K-Means
barely moves. The pairs have to be held in memory, which is the real wall.

**Hierarchical clustering is a tool for thousands of rows, not millions.**

---
# 3 · DBSCAN — clustering by density

A completely different idea. K-Means asks *"which centre are you nearest?"*. DBSCAN asks
**"are you in a crowd?"** — and that one change is what frees it from round shapes.

Two settings:

- **`eps`** — the radius of the circle you draw around each point
- **`min_samples`** — how many neighbours inside that circle count as a crowd

Every point then falls into one of three kinds.

In [ ]:
# two blobs plus genuinely scattered outliers
rg = np.random.default_rng(44)
P = np.vstack([rg.normal([0, 0], 0.55, (70, 2)),
               rg.normal([3.6, 2.4], 0.55, (60, 2)),
               rg.uniform([-2.2, -2.2], [6.0, 4.8], (18, 2))])

EPS, MIN_SAMPLES = 0.55, 5

# --- work out the three kinds yourself, before using sklearn ---------
D = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=2))
n_within = (D <= EPS).sum(axis=1)          # includes the point itself
is_core = n_within >= MIN_SAMPLES
# a border point is not core, but sits inside some core point's circle
is_border = (~is_core) & ((D <= EPS)[:, is_core].any(axis=1))
is_noise = (~is_core) & (~is_border)

print(f"core   {is_core.sum():4d}")
print(f"border {is_border.sum():4d}")
print(f"noise  {is_noise.sum():4d}")
print(f"total  {is_core.sum() + is_border.sum() + is_noise.sum():4d}  of {len(P)}")

In [ ]:
db = DBSCAN(eps=EPS, min_samples=MIN_SAMPLES).fit(P)
sk_core = np.zeros(len(P), bool); sk_core[db.core_sample_indices_] = True

print("my core points match sklearn's:", bool((sk_core == is_core).all()))
print("my noise points match sklearn's:", bool(((db.labels_ == -1) == is_noise).all()))
print()
print("clusters DBSCAN found:", len(set(db.labels_)) - (1 if -1 in db.labels_ else 0))
print("...and nobody told it there were two.")

fig, ax = plt.subplots(figsize=(6.4, 4.8))
ax.scatter(*P[is_core].T, c=TEAL, s=34, edgecolors="white", linewidths=.5, label="core")
ax.scatter(*P[is_border].T, c=AMBER, s=34, edgecolors="white", linewidths=.5, label="border")
ax.scatter(*P[is_noise].T, c=CORAL, marker="x", s=46, linewidths=2, label="noise")
ax.legend(fontsize=9, frameon=False); ax.set_xticks([]); ax.set_yticks([])
ax.set_title("core, border and noise")
plt.show()

**That third category is unique.** Every other method today forces every point into a
cluster. DBSCAN is allowed to say *"this one belongs nowhere"*, which makes it an
outlier detector as much as a clustering method.

> Note that only **10** of the 18 points we scattered on purpose got labelled noise. The
> other 8 happened to land inside a blob — and DBSCAN is right to absorb them. "Outlier"
> is a fact about *position*, not about how the point was generated.

## 3.1 · eps is the whole model

In [ ]:
print(f"{'eps':>6}{'clusters':>10}{'noise':>8}{'agreement':>12}")
for eps in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    d = DBSCAN(eps=eps, min_samples=5).fit(XM)
    nc = len(set(d.labels_)) - (1 if -1 in d.labels_ else 0)
    print(f"{eps:>6}{nc:>10}{(d.labels_ == -1).sum():>8}"
          f"{adjusted_rand_score(YM, d.labels_):>12.4f}")

There is no gentle failure here:

- **0.05** — almost nothing has enough neighbours, so 243 of 300 points are noise
- **0.10** — the crescents shatter into eleven fragments
- **0.15 – 0.20** — two clusters, agreement 1.000
- **0.30 and up** — the gap between the crescents gets bridged and everything fuses into
  one cluster, and agreement collapses to **0.000**

## 3.2 · The standard recipe for choosing eps

Measure every point's distance to its 5th nearest neighbour, sort them, and look for the
elbow.

In [ ]:
dist, _ = NearestNeighbors(n_neighbors=5).fit(XM).kneighbors(XM)
kd = np.sort(dist[:, -1])

plt.figure(figsize=(6, 3.2))
plt.plot(kd, color=BLUE, lw=2)
plt.axhline(np.quantile(kd, .90), color=CORAL, ls="--")
plt.text(2, np.quantile(kd, .90) + .006, f"90th percentile = {np.quantile(kd, .90):.3f}",
         color=CORAL, fontsize=9, fontweight="bold")
plt.xlabel("points, sorted"); plt.ylabel("distance to 5th neighbour")
plt.title("the k-distance plot"); plt.tight_layout(); plt.show()

suggested = float(np.quantile(kd, .90))
for e in [suggested, 0.20]:
    d = DBSCAN(eps=e, min_samples=5).fit(XM)
    print(f"  eps = {e:.3f}  ->  agreement {adjusted_rand_score(YM, d.labels_):.4f}")

**The recipe suggests about 0.11, which scores 0.337. The value that actually works is
0.20, scoring 1.000.**

The k-distance plot gives you the right order of magnitude, not the answer. Expect to try
a handful of values around it — and to plot the result each time.

## 3.3 · Where DBSCAN falls over

DBSCAN uses **one radius for the whole dataset**. If one group is tight and another is
spread out, no single value can suit both.

In [ ]:
r2 = np.random.default_rng(5)
XD = np.vstack([r2.normal([0, 0], 0.30, (120, 2)),      # tight
                r2.normal([4.2, 0], 1.25, (120, 2))])   # four times looser
YD = np.r_[np.zeros(120), np.ones(120)].astype(int)

print(f"{'eps':>6}{'clusters':>10}{'noise':>8}{'agreement':>12}")
for eps in [0.25, 0.45, 0.80]:
    d = DBSCAN(eps=eps, min_samples=5).fit(XD)
    nc = len(set(d.labels_)) - (1 if -1 in d.labels_ else 0)
    print(f"{eps:>6}{nc:>10}{(d.labels_ == -1).sum():>8}"
          f"{adjusted_rand_score(YD, d.labels_):>12.4f}")

km_d = KMeans(2, n_init=10, random_state=0).fit(XD)
print(f"\n  plain K-Means, k=2:  {adjusted_rand_score(YD, km_d.labels_):.4f}")

Look at the **noise** column. To be tight enough to hold the dense group together,
`eps = 0.25` has to throw away **117 of the 240 points**. Loosen it and the dense group
starts swallowing its surroundings.

**And plain K-Means beats every setting.** This is worth saying out loud: the newest
method is not the best method. DBSCAN is better than K-Means on *shape* and worse on
*varying density*, and the only way to know which matters for your data is to look at it.

---
# 4 · Four shapes, four methods

Same data, same scoring, no favourites. DBSCAN's `eps` is chosen by the k-distance recipe
each time — **without looking at the answers**, which is the only fair way to do it.

In [ ]:
def eps_from_kdistance(D, min_samples=5, q=0.90):
    d, _ = NearestNeighbors(n_neighbors=min_samples).fit(D).kneighbors(D)
    return float(np.quantile(np.sort(d[:, -1]), q))

XC, YC = make_circles(300, factor=0.45, noise=0.05, random_state=0)
XB, YB = make_blobs(300, centers=3, cluster_std=1.0, random_state=7)
SETS = [("blobs", XB, YB, 3), ("moons", XM, YM, 2),
        ("circles", XC, YC, 2), ("uneven density", XD, YD, 2)]

rows = {}
fig, axes = plt.subplots(4, 4, figsize=(13, 11))
for r, m in enumerate(["K-Means", "Ward", "Single link", "DBSCAN"]):
    for c, (nm, D, y, k) in enumerate(SETS):
        Zs = StandardScaler().fit_transform(D)
        if m == "K-Means":
            lab = KMeans(k, n_init=10, random_state=0).fit(Zs).labels_
        elif m == "Ward":
            lab = AgglomerativeClustering(k, linkage="ward").fit(Zs).labels_
        elif m == "Single link":
            lab = AgglomerativeClustering(k, linkage="single").fit(Zs).labels_
        else:
            lab = DBSCAN(eps=eps_from_kdistance(Zs), min_samples=5).fit(Zs).labels_
        a = adjusted_rand_score(y, lab)
        rows.setdefault(m, {})[nm] = round(a, 4)
        show(Zs, lab, f"{m} on {nm}:  {a:.3f}", ax=axes[r][c])
plt.tight_layout(); plt.show()

print(pd.DataFrame(rows).T)

**There is no winning row.**

- **K-Means** wins the blobs, and scores about **zero** on circles
- **Ward** matches it on blobs, wins uneven density, and is also useless on circles
- **Single link** scores a perfect **1.000** on blobs, moons *and* circles — then
  **0.000** on uneven density, because one bridge of points chained everything together
- **DBSCAN** is respectable everywhere and best nowhere, because we chose `eps` by recipe
  rather than tuning it

That last row is worth sitting with. With `eps` tuned by hand DBSCAN scored 1.000 on the
moons; with `eps` chosen honestly it scored 0.737. **Most of DBSCAN's reputation depends
on getting `eps` right, and getting `eps` right is the hard part.**

---
# 5 · Summary

| | |
|---|---|
| **Agglomerative** | Every point starts alone; merge the closest pair; `n − 1` merges in total. |
| **The dendrogram** | The record of every merge, at the height it happened. Cut it anywhere for any `k`. |
| **Where to cut** | Inside the biggest vertical gap. Here the 3→2 jump cost 12.54, the largest by far. |
| **Linkage** | Group-to-group distance. Ward by default; single follows curves and chains through noise. |
| **The cost** | `n²` pairs in memory — 7,998,000 at just 4,000 rows. Thousands, not millions. |
| **DBSCAN** | Core, border, noise. Clusters grow through crowds, so they take any shape. |
| **Free with it** | It finds `k` itself, and may label a point `−1`. No other method can decline. |
| **Its weakness** | One `eps` for everything, so clusters of different density break it. |
| **Honestly** | No method won all four shapes. Plot the data; its shape chooses the algorithm. |

### Next: Day 45

Clustering found groups in two columns, where you could see the answer. Real data has
thirty columns and cannot be plotted at all. **PCA** squashes many columns down to two
or three that carry most of the variation — so you can finally look at it.